In [21]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

       

load_dotenv('/Users/dineshjadhav/Desktop/genai-course/openai_key.env', override=True) # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )
if api_key:
    pretty_print(f"Key loaded. Length: {len(api_key)}, ends with: ...{api_key[-6:]}")
pretty_print("API key loaded successfully.")

Key loaded. Length: 164, ends with: ...I8H58A
API key loaded successfully.


In [22]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

MODEL  = "gpt-5-nano"  

OpenAI client ready.


In [ ]:
SYSTEM_PROMPT = """You are a coding agent. The user will give you a problem \
statement written in natural language. Your job:
 
1. Determine which programming language the user wants the solution in.
   - If a language is explicitly named, use that one.
   - If it's only hinted at (framework names, syntax, file extension), \
infer it from context.
   - If no language is specified or implied at all, default to Python.
2. Write a complete, correct, well-commented solution to the problem in \
that language.
3. Briefly explain your solution (2-4 sentences).
 
Respond with ONLY a single JSON object, no markdown fences, no extra text, \
matching exactly this schema:
 
{
  "language": "<human readable name, e.g. 'Python', 'Java'>",
  "code": "<the complete source code as a single string, with real \\n newlines>",
  "explanation": "<short explanation of the approach>"
}
"""
    
 
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
 
def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception:
        print(text)  # fallback to normal print if text isn't a plain string
 

def call_llm(problem_prompt: str, model: str = "gpt-4o-mini") -> dict:
    """Calls the LLM and returns a parsed dict: language, file_extension, code, explanation."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem_prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        sys.exit(f"Model did not return valid JSON:\n{raw}")
 
 
def run_agent(problem_prompt: str, model: str = "gpt-4o-mini"):
    resp = call_llm(problem_prompt, model)
 
    language = resp.get("language", "Unknown")
    code = resp.get("code", "")
    explanation = resp.get("explanation", "")
 
    pretty_print("Detected language:", language)
    pretty_print("Explanation:", explanation)
 
    print("\nGenerated code:\n")
    print(code)
 
    return resp
 

In [39]:
response = run_agent(
    problem_prompt="Write a function that takes a list of integers and returns the sum of all even numbers in the list."
)


Detected language: Python
Explanation: The function 'sum_of_evens' iterates through the provided list of
integers, checking each number to see if it is even. If a number is even, it
adds it to a running total, which is returned at the end. This approach ensures
that only even numbers contribute to the final sum.

Generated code:

def sum_of_evens(numbers):
    """Returns the sum of all even numbers in the given list."""
    total = 0  # Initialize total sum to 0
    for num in numbers:  # Iterate through each number in the list
        if num % 2 == 0:  # Check if the number is even
            total += num  # Add even number to total
    return total  # Return the final sum of even numbers

# Example usage:
# print(sum_of_evens([1, 2, 3, 4, 5, 6]))  # Output: 12


In [ ]:
#!/usr/bin/env python3
"""
test_case_agent.py

An agent that generates test cases for code produced by coding_agent.py.

Workflow:
  1. Take a problem statement + the language + the generated code.
  2. Ask the LLM to write a runnable test suite (using the standard
     testing convention for that language, e.g. pytest for Python,
     JUnit for Java, Jest for JavaScript, etc.).
  3. Pretty-print an explanation of the test coverage, then print the
     test code.

Can be used standalone, or chained with coding_agent.run_agent() to go
straight from "problem statement" -> "code" -> "tests" in one run.

Usage:
    python test_case_agent.py "Write a function to check if a number is prime, in Python"

Requires:
    pip install openai python-dotenv --break-system-packages
"""

import sys
import json

# Reuse the already-configured client, pretty_print, and code-generation
# agent from coding_agent.py so API key loading isn't duplicated.
from coding_agent import client, pretty_print, run_agent


TEST_SYSTEM_PROMPT = """You are a test-writing agent. The user will give you:
  - A problem statement
  - The programming language of the solution
  - The solution code itself

Your job:
1. Write a runnable test suite for the given code, using the standard/idiomatic \
testing convention for that language (e.g. pytest for Python, JUnit for Java, \
Jest/Mocha for JavaScript, Go's built-in "testing" package for Go, etc.).
2. Cover: typical/expected cases, edge cases (empty input, zero, negative \
numbers, boundaries, etc. as relevant), and at least one invalid-input case \
if the problem allows for it.
3. Briefly explain what the test suite covers (2-4 sentences).

Respond with ONLY a single JSON object, no markdown fences, no extra text, \
matching exactly this schema:

{
  "test_code": "<the complete test suite as a single string, with real \\n newlines>",
  "explanation": "<short explanation of what is covered and why>"
}
"""


def generate_test_cases(problem_prompt: str, language: str, code: str,
                         model: str = "gpt-4o-mini") -> dict:
    """Calls the LLM to generate a test suite for the given code."""
    user_content = (
        f"Problem statement:\n{problem_prompt}\n\n"
        f"Language: {language}\n\n"
        f"Solution code:\n{code}"
    )
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": TEST_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        sys.exit(f"Model did not return valid JSON:\n{raw}")


def run_test_agent(problem_prompt: str, language: str, code: str,
                    model: str = "gpt-4o-mini") -> dict:
    resp = generate_test_cases(problem_prompt, language, code, model)

    explanation = resp.get("explanation", "")
    test_code = resp.get("test_code", "")

    pretty_print("Test coverage:", explanation)
    print("\nGenerated test code:\n")
    print(test_code)

    return resp


def solve_and_test(problem_prompt: str, model: str = "gpt-4o-mini"):
    """Chains coding_agent -> test_case_agent: problem -> code -> tests."""
    code_resp = run_agent(problem_prompt, model)
    print("\n" + "=" * 60 + "\n")
    test_resp = run_test_agent(
        problem_prompt,
        code_resp.get("language", "Python"),
        code_resp.get("code", ""),
        model,
    )
    return code_resp, test_resp


Detected language: Python
Explanation: The function `is_prime` checks if a number is prime by first ruling
out numbers less than or equal to 1. It then checks for divisibility from 2 up
to the square root of the number, returning False if any divisors are found, and
True if none are found.
Explanation: def is_prime(n):     """Check if a number is prime."""     if n <=
1:         return False  # 0 and 1 are not prime numbers     for i in range(2,
int(n**0.5) + 1):         if n % i == 0:             return False  # Found a
divisor, not prime     return True  # No divisors found, it is prime  # Example
usage: print(is_prime(11))  # Output: True print(is_prime(4))   # Output: False
